# Week 1 — Linear Regression from Scratch (NumPy)

**Goal:** Understand linear regression by implementing it yourself — no sklearn for the core model.

We'll:
1. Generate synthetic data
2. Define the model, MSE loss, and gradients
3. Train with **batch gradient descent**
4. Plot the fit and loss curve
5. Compare to a closed-form / `numpy.linalg.lstsq` solution

## Intuition

A simple linear model predicts:

$$\hat{y} = w x + b$$

- **\(w\)** (weight / slope): how much \(y\) changes when \(x\) increases by 1
- **\(b\)** (bias / intercept): the predicted value when \(x = 0\)

We measure error with **mean squared error (MSE)**:

$$L = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2$$

**Gradient descent** nudges \(w\) and \(b\) opposite the gradient of \(L\), so the loss goes downhill each step:

$$w \leftarrow w - \eta \frac{\partial L}{\partial w},\quad b \leftarrow b - \eta \frac{\partial L}{\partial b}$$

where \(\eta\) is the learning rate.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Synthetic data

We invent a true line \(y = 2.5x - 1\) and add Gaussian noise so learning is nontrivial.

In [ ]:
def make_synthetic_data(
    n=100, true_w=2.5, true_b=-1.0, noise_std=0.8, seed=42
):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-3.0, 3.0, size=n)
    noise = rng.normal(0.0, noise_std, size=n)
    y = true_w * x + true_b + noise
    return x, y

true_w, true_b = 2.5, -1.0
x, y = make_synthetic_data(n=100, true_w=true_w, true_b=true_b, noise_std=0.8)

plt.figure(figsize=(6, 4))
plt.scatter(x, y, alpha=0.6, edgecolors="none")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Synthetic data")
plt.show()
print(f"n={len(x)}, true_w={true_w}, true_b={true_b}")

## 2. Model, loss, and gradients

For MSE, the gradients are:

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_i (\hat{y}_i - y_i)\, x_i$$

$$\frac{\partial L}{\partial b} = \frac{2}{n}\sum_i (\hat{y}_i - y_i)$$

In [ ]:
def predict(x, w, b):
    return w * x + b

def mse(y_true, y_pred):
    return float(np.mean((y_pred - y_true) ** 2))

def gradients(x, y, w, b):
    y_hat = predict(x, w, b)
    err = y_hat - y
    n = len(y)
    dw = (2.0 / n) * np.sum(err * x)
    db = (2.0 / n) * np.sum(err)
    return float(dw), float(db)

# Quick sanity check at a random starting point
dw0, db0 = gradients(x, y, 0.0, 0.0)
print(f"Gradients at (w=0, b=0): dw={dw0:.4f}, db={db0:.4f}")

## 3. Train with batch gradient descent

Each epoch: compute gradients on the **full** dataset, then update \(w\) and \(b\).

In [ ]:
def train_gd(x, y, lr=0.05, epochs=200, w0=0.0, b0=0.0):
    w, b = w0, b0
    history = []
    for _ in range(epochs):
        dw, db = gradients(x, y, w, b)
        w -= lr * dw
        b -= lr * db
        history.append(mse(y, predict(x, w, b)))
    return w, b, history

w_gd, b_gd, history = train_gd(x, y, lr=0.05, epochs=200)
print(f"GD result: w={w_gd:.4f}, b={b_gd:.4f}, final MSE={history[-1]:.4f}")
print(f"True:      w={true_w:.4f}, b={true_b:.4f}")

## 4. Closed-form comparison (`numpy.linalg.lstsq`)

For this linear model we can also solve the least-squares problem directly.
Gradient descent should land very close to this solution (given enough epochs / a good learning rate).

In [ ]:
def closed_form_lstsq(x, y):
    A = np.column_stack([x, np.ones_like(x)])
    params, *_ = np.linalg.lstsq(A, y, rcond=None)
    return float(params[0]), float(params[1])

w_cf, b_cf = closed_form_lstsq(x, y)
print(f"lstsq:     w={w_cf:.4f}, b={b_cf:.4f}, MSE={mse(y, predict(x, w_cf, b_cf)):.4f}")
print(f"GD:        w={w_gd:.4f}, b={b_gd:.4f}, MSE={history[-1]:.4f}")
print(f"|Δw|={abs(w_gd - w_cf):.6f}, |Δb|={abs(b_gd - b_cf):.6f}")

## 5. Plots — fit and loss curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

xs = np.linspace(x.min(), x.max(), 200)
axes[0].scatter(x, y, alpha=0.6, label="data", edgecolors="none")
axes[0].plot(xs, predict(xs, w_gd, b_gd), color="C1", lw=2, label="GD fit")
axes[0].plot(xs, predict(xs, w_cf, b_cf), color="C2", lw=2, ls="--", label="lstsq fit")
axes[0].plot(xs, predict(xs, true_w, true_b), color="C3", lw=1.5, ls=":", label="true line")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_title("Data and fitted lines")
axes[0].legend()

axes[1].plot(history, color="C0")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("MSE")
axes[1].set_title("Training loss (gradient descent)")
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Takeaways

- Linear regression = fit a line (or hyperplane) by minimizing MSE.
- Gradients tell you which way to move \(w\) and \(b\); the learning rate controls step size.
- Batch GD on a convex MSE problem converges to (near) the same answer as closed-form least squares.
- Next week ideas: multiple features, polynomial features, or stochastic / mini-batch GD.